# SEG Multi-Seed Benchmark — BACE

Kompaktes Notebook für Experimente mit verschiedenen Seeds.

**Workflow:**
1. Zellen 1-4 einmal ausführen (Setup, Daten, Embeddings, Funktionen)  
2. Config definieren und `run_single_seed(config, seed)` aufrufen  
3. Oder: `run_multi_seed(config, seeds=[...])` für aggregierte Ergebnisse

In [1]:
# === Setup (einmal ausführen) ===
import sys
import random
from pathlib import Path
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import numpy as np
import pandas as pd
import torch
import deepchem as dc
from typing import Dict, List, Any, Optional

print(f"Workspace: {workspace_root}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Users\robsc\Home\Dev\molfusion2\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric depe

Workspace: c:\Users\robsc\Home\Dev\molfusion2
PyTorch: 2.6.0+cu124, CUDA: True


In [2]:
# === Default Configuration ===
DEFAULT_CONFIG = {
    # Architecture
    "hidden_channels": 64,
    "K": 3,
    "num_layers": 2,
    "pool": "sum",
    "set2set_processing_steps": 6,
    
    # Fusion
    "fusion": "cross_mha",
    "fusion_dim": 64,
    "fusion_n_heads": 8,
    "text_proj_init": "xavier",
    "text_proj_init_gain": 0.1,
    "freeze_text_proj": True,
    
    # Regularization
    "dropout": 0.3,
    "fusion_dropout": 0.3,
    "head_dropout": 0.5,
    "weight_decay": 1e-1,
    
    # Head
    "head_type": "mlp",
    "head_hidden_dim": 32,
    
    # Training
    "learning_rate": 5e-4,
    "batch_size": 32,
    "num_epochs": 200,
    "patience": 20,
    "scheduler": "cosine",
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    "grad_clip": None,
}

print("Default config loaded.")

Default config loaded.


In [3]:
# === Load BACE Dataset (einmal ausführen) ===
tasks, datasets, _ = dc.molnet.load_bace_classification(featurizer='ECFP', splitter='scaffold')
train_dc, valid_dc, test_dc = datasets

TRAIN_SMILES = list(train_dc.ids)
TRAIN_Y = train_dc.y.reshape(-1).astype(np.float32)
VALID_SMILES = list(valid_dc.ids)
VALID_Y = valid_dc.y.reshape(-1).astype(np.float32)
TEST_SMILES = list(test_dc.ids)
TEST_Y = test_dc.y.reshape(-1).astype(np.float32)

print(f"Train: {len(TRAIN_SMILES)} ({TRAIN_Y.mean()*100:.1f}% active) | Valid: {len(VALID_SMILES)} | Test: {len(TEST_SMILES)}")

Train: 1210 (42.6% active) | Valid: 151 | Test: 152


In [4]:
# === Load Text Embeddings (einmal ausführen) ===
from utils.embedding_cache import EfficientEmbeddingCache

COT_EMB_DIR = workspace_root / "cache" / "cot_embeddings"
TASK = "binding_fast"

npz_path = COT_EMB_DIR / f"{TASK}_text_embeddings_compact.npz"
cache = EfficientEmbeddingCache.load(npz_path)

all_smiles = TRAIN_SMILES + VALID_SMILES + TEST_SMILES
all_emb = torch.from_numpy(cache.get_batch(all_smiles))

n_train, n_valid = len(TRAIN_SMILES), len(VALID_SMILES)
TRAIN_TEXT_EMB = all_emb[:n_train]
VALID_TEXT_EMB = all_emb[n_train:n_train + n_valid]
TEST_TEXT_EMB = all_emb[n_train + n_valid:]

print(f"\u2713 Loaded {npz_path.name} ({len(cache)} molecules)")
print(f"  Embeddings: train={TRAIN_TEXT_EMB.shape}, valid={VALID_TEXT_EMB.shape}, test={TEST_TEXT_EMB.shape}")

Loading embeddings from binding_fast_text_embeddings_compact.npz...
  Loaded 1513 entries, dim=3072
  Memory mode: mapped
✓ Loaded binding_fast_text_embeddings_compact.npz (1513 molecules)
  Embeddings: train=torch.Size([1210, 3072]), valid=torch.Size([151, 3072]), test=torch.Size([152, 3072])


In [5]:
# === Experiment Functions (einmal ausführen) ===
from itertools import product
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from models import SEGPredictor, SEGPredictorConfig


def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def classification_metrics(y_true, y_pred_proba, threshold=0.5) -> Dict[str, float]:
    """Calculate AUC-ROC, Accuracy, F1."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred_proba = np.clip(np.asarray(y_pred_proba).reshape(-1), 0, 1)
    mask = np.isfinite(y_pred_proba)
    y_true, y_pred_proba = y_true[mask], y_pred_proba[mask]
    y_pred = (y_pred_proba >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, y_pred_proba)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


def run_single_seed(
    config: Dict[str, Any],
    seed: int,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run a single SEG experiment with the given config and seed.
    
    Args:
        config: Model/training config (merged with DEFAULT_CONFIG)
        seed: Random seed for reproducibility
        verbose: Print training progress
    
    Returns:
        Dict with {seed, metrics, history, model}
    """
    cfg = {**DEFAULT_CONFIG, **config}
    set_seed(seed)
    
    seg_config = SEGPredictorConfig(
        task="classification",
        hidden_channels=cfg["hidden_channels"],
        K=cfg["K"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        pool=cfg["pool"],
        set2set_processing_steps=cfg["set2set_processing_steps"],
        text_embedding_dim=3072,
        text_projection_dim=cfg["fusion_dim"],
        text_proj_init=cfg["text_proj_init"],
        text_proj_init_gain=cfg["text_proj_init_gain"],
        freeze_text_proj=cfg["freeze_text_proj"],
        fusion=cfg["fusion"],
        fusion_dim=cfg["fusion_dim"],
        fusion_n_heads=cfg.get("fusion_n_heads", 8),
        fusion_dropout=cfg["fusion_dropout"],
        head_type=cfg["head_type"],
        head_hidden_dim=cfg["head_hidden_dim"],
        head_dropout=cfg["head_dropout"],
    )
    
    seg = SEGPredictor(config=seg_config)
    
    if verbose:
        print(f"=== Seed {seed} ===")
    
    history = seg.fit(
        smiles_list=TRAIN_SMILES,
        labels=TRAIN_Y.tolist(),
        val_smiles=VALID_SMILES,
        val_labels=VALID_Y.tolist(),
        text_embeddings=TRAIN_TEXT_EMB,
        val_text_embeddings=VALID_TEXT_EMB,
        num_epochs=cfg["num_epochs"],
        batch_size=cfg["batch_size"],
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        patience=cfg["patience"],
        scheduler=cfg["scheduler"],
        scheduler_patience=cfg["scheduler_patience"],
        scheduler_factor=cfg["scheduler_factor"],
        min_lr=cfg["min_lr"],
        grad_clip=cfg["grad_clip"],
        seed=seed,
        verbose=verbose,
    )
    
    preds = seg.predict_batch(TEST_SMILES, text_embeddings=TEST_TEXT_EMB)
    metrics = classification_metrics(TEST_Y, preds)
    
    if verbose:
        print(f"\u2192 Test AUC={metrics['roc_auc']:.4f}, Acc={metrics['accuracy']:.4f}, F1={metrics['f1']:.4f}\n")
    
    return {"seed": seed, "metrics": metrics, "history": history, "model": seg}


def run_multi_seed(
    config: Dict[str, Any],
    seeds: List[int],
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Run experiments with multiple seeds and aggregate results.
    """
    results = []
    
    print(f"Running {len(seeds)} experiments with seeds: {seeds}")
    print("-" * 60)
    
    for i, seed in enumerate(seeds):
        print(f"[{i+1}/{len(seeds)}] Seed {seed}...", end=" ", flush=True)
        result = run_single_seed(config, seed, verbose=verbose)
        results.append(result)
        if not verbose:
            m = result["metrics"]
            print(f"AUC={m['roc_auc']:.4f}, Acc={m['accuracy']:.4f}, F1={m['f1']:.4f}")
    
    df = pd.DataFrame([
        {"seed": r["seed"], **r["metrics"]}
        for r in results
    ])
    
    summary = {
        "roc_auc_mean": df["roc_auc"].mean(),
        "roc_auc_std": df["roc_auc"].std(),
        "accuracy_mean": df["accuracy"].mean(),
        "accuracy_std": df["accuracy"].std(),
        "f1_mean": df["f1"].mean(),
        "f1_std": df["f1"].std(),
        "n_runs": len(seeds),
        "seeds": seeds,
    }
    
    print("-" * 60)
    print(f"\n=== AGGREGATED RESULTS ({len(seeds)} seeds) ===")
    print(f"AUC-ROC:  {summary['roc_auc_mean']:.4f} \u00b1 {summary['roc_auc_std']:.4f}")
    print(f"Accuracy: {summary['accuracy_mean']:.4f} \u00b1 {summary['accuracy_std']:.4f}")
    print(f"F1:       {summary['f1_mean']:.4f} \u00b1 {summary['f1_std']:.4f}")
    
    return {"summary": summary, "runs": results, "df": df}


def run_grid_search(
    grid_config: Dict[str, Any],
    seeds: List[int] = [42],
    sort_by: str = "roc_auc",
    ascending: bool = False,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Grid search over hyperparameters with multi-seed evaluation.
    
    Values that are lists \u2192 swept over (all combinations).
    Values that are scalars \u2192 fixed for all runs.
    
    Example:
        grid_config = {
            "dropout": [0.2, 0.3, 0.5],
            "weight_decay": [1e-2, 1e-1],
            "fusion": "cross_mha",           # fixed
        }
        df = run_grid_search(grid_config, seeds=[42, 123])
    """
    sweep_keys = []
    sweep_values = []
    fixed_params = {}
    
    for k, v in grid_config.items():
        if isinstance(v, list):
            sweep_keys.append(k)
            sweep_values.append(v)
        else:
            fixed_params[k] = v
    
    if sweep_keys:
        combos = list(product(*sweep_values))
    else:
        combos = [()]
    
    n_combos = len(combos)
    n_total = n_combos * len(seeds)
    
    print(f"=== GRID SEARCH ===")
    if sweep_keys:
        print(f"Sweep params: {', '.join(f'{k} ({len(v)} values)' for k, v in zip(sweep_keys, sweep_values))}")
    else:
        print("No sweep params (single config)")
    print(f"Combinations: {n_combos} \u00d7 {len(seeds)} seeds = {n_total} total runs")
    print("=" * 70)
    
    all_rows = []
    run_counter = 0
    
    for combo_idx, combo in enumerate(combos):
        config = {**fixed_params}
        for k, v in zip(sweep_keys, combo):
            config[k] = v
        
        combo_desc = ", ".join(f"{k}={v}" for k, v in zip(sweep_keys, combo)) if sweep_keys else "default"
        print(f"\n[{combo_idx+1}/{n_combos}] {combo_desc}")
        
        seed_metrics = []
        for seed in seeds:
            run_counter += 1
            print(f"  ({run_counter}/{n_total}) seed={seed}...", end=" ", flush=True)
            result = run_single_seed(config, seed, verbose=verbose)
            seed_metrics.append(result["metrics"])
            m = result["metrics"]
            print(f"AUC={m['roc_auc']:.4f}")
        
        aucs = [m["roc_auc"] for m in seed_metrics]
        accs = [m["accuracy"] for m in seed_metrics]
        f1s = [m["f1"] for m in seed_metrics]
        
        row = {**{k: v for k, v in zip(sweep_keys, combo)}}
        row["roc_auc_mean"] = np.mean(aucs)
        row["roc_auc_std"] = np.std(aucs)
        row["accuracy_mean"] = np.mean(accs)
        row["accuracy_std"] = np.std(accs)
        row["f1_mean"] = np.mean(f1s)
        row["f1_std"] = np.std(f1s)
        row["n_seeds"] = len(seeds)
        all_rows.append(row)
    
    df = pd.DataFrame(all_rows)
    
    sort_col = f"{sort_by}_mean"
    if sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
    
    print("\n" + "=" * 70)
    print(f"GRID SEARCH RESULTS (sorted by {sort_by})")
    print("=" * 70)
    
    for i, row in df.iterrows():
        params = " | ".join(f"{k}={row[k]}" for k in sweep_keys) if sweep_keys else "default"
        print(f"  #{i+1}: {params}")
        print(f"      AUC={row['roc_auc_mean']:.4f}\u00b1{row['roc_auc_std']:.4f}  "
              f"Acc={row['accuracy_mean']:.4f}\u00b1{row['accuracy_std']:.4f}  "
              f"F1={row['f1_mean']:.4f}\u00b1{row['f1_std']:.4f}")
    
    return df


print("\u2713 Functions loaded: run_single_seed, run_multi_seed, run_grid_search")

✓ Functions loaded: run_single_seed, run_multi_seed, run_grid_search


---
## Experimente

Ab hier: Config definieren und Experimente starten.

In [6]:
# === Einzelnes Experiment ===
# Config-Overrides (leer = Default-Config verwenden)
my_config = {
    # Hier eigene Werte überschreiben, z.B.:
    # "dropout": 0.5,
    # "weight_decay": 5e-2,
}

result = run_single_seed(my_config, seed=42)

=== Seed 42 ===
Pre-computing molecular graphs for 1210 molecules...
Pre-computed 1210/1210 valid graphs
Pre-computing molecular graphs for 151 molecules...
Pre-computed 151/151 valid graphs
Training SEGPredictor on 1210 molecules...
Task: classification
Validation set: 151 molecules
Fusion method: cross_mha
LR scheduler: cosine
Epoch 001 | Train Loss: 0.6903 | Val BCE: 0.7044 | LR: 5.00e-04
Epoch 005 | Train Loss: 0.6525 | Val BCE: 0.7768 | LR: 4.99e-04
Epoch 010 | Train Loss: 0.5073 | Val BCE: 0.8748 | LR: 4.97e-04
Epoch 015 | Train Loss: 0.4776 | Val BCE: 0.7753 | LR: 4.93e-04
Epoch 020 | Train Loss: 0.4323 | Val BCE: 0.6493 | LR: 4.88e-04
Epoch 025 | Train Loss: 0.4043 | Val BCE: 0.6881 | LR: 4.81e-04
Epoch 030 | Train Loss: 0.4067 | Val BCE: 0.7617 | LR: 4.73e-04
Epoch 035 | Train Loss: 0.3609 | Val BCE: 0.7369 | LR: 4.63e-04
Epoch 040 | Train Loss: 0.3513 | Val BCE: 0.8738 | LR: 4.52e-04
Epoch 045 | Train Loss: 0.3450 | Val BCE: 0.9011 | LR: 4.40e-04
Epoch 050 | Train Loss: 0.342

In [7]:
# === Multi-Seed Experiment ===
my_config = {}  # Default-Config

results = run_multi_seed(my_config, seeds=[42, 123, 456, 789, 1337])

Running 5 experiments with seeds: [42, 123, 456, 789, 1337]
------------------------------------------------------------
[1/5] Seed 42... AUC=0.8324, Acc=0.7237, F1=0.7470
[2/5] Seed 123... AUC=0.8870, Acc=0.8487, F1=0.8729
[3/5] Seed 456... AUC=0.8404, Acc=0.7829, F1=0.8254
[4/5] Seed 789... AUC=0.8804, Acc=0.7500, F1=0.7625
[5/5] Seed 1337... AUC=0.8755, Acc=0.7895, F1=0.8182
------------------------------------------------------------

=== AGGREGATED RESULTS (5 seeds) ===
AUC-ROC:  0.8632 ± 0.0249
Accuracy: 0.7789 ± 0.0471
F1:       0.8052 ± 0.0509


In [8]:
# === Ergebnisse anzeigen ===
results["df"]

,seed,roc_auc,accuracy,f1
0,42,0.832428,0.723684,0.746988
1,123,0.886957,0.848684,0.872928
2,456,0.840399,0.782895,0.825397
3,789,0.880435,0.750000,0.762500
4,1337,0.875543,0.789474,0.818182


---
## Varianten testen

Config anpassen und erneut ausführen:

In [20]:
# === Grid Search Beispiel ===
# Werte als Liste → werden gesweept (alle Kombinationen)
# Werte als Skalar → bleiben fix
grid_config = {
    "hidden_channels" : [128],
    "K" : [3,4],
    "num_layers" : [2,3,4],
    "text_proj_init": ["xavier"],
    "pool": ["sum"],
    "freeze_text_proj": [True],
    "fusion_dim": [32],
    "fusion": "cross_mha",       # fix
    "head_type": "mlp",          # fix
}

# Grid search mit 4 Seeds pro Kombination
grid_df = run_grid_search(grid_config, seeds=[42, 113, 1235])

=== GRID SEARCH ===
Sweep params: hidden_channels (1 values), K (2 values), num_layers (3 values), text_proj_init (1 values), pool (1 values), freeze_text_proj (1 values), fusion_dim (1 values)
Combinations: 6 × 3 seeds = 18 total runs

[1/6] hidden_channels=128, K=3, num_layers=2, text_proj_init=xavier, pool=sum, freeze_text_proj=True, fusion_dim=32
  (1/18) seed=42... AUC=0.8665
  (2/18) seed=113... AUC=0.8938
  (3/18) seed=1235... AUC=0.8665

[2/6] hidden_channels=128, K=3, num_layers=3, text_proj_init=xavier, pool=sum, freeze_text_proj=True, fusion_dim=32
  (4/18) seed=42... AUC=0.8803
  (5/18) seed=113... AUC=0.8784
  (6/18) seed=1235... AUC=0.8801

[3/6] hidden_channels=128, K=3, num_layers=4, text_proj_init=xavier, pool=sum, freeze_text_proj=True, fusion_dim=32
  (7/18) seed=42... AUC=0.8752
  (8/18) seed=113... AUC=0.8596
  (9/18) seed=1235... AUC=0.8589

[4/6] hidden_channels=128, K=4, num_layers=2, text_proj_init=xavier, pool=sum, freeze_text_proj=True, fusion_dim=32
  (10/18

In [21]:
# === Grid Search Ergebnisse ===
grid_df

,hidden_channels,K,num_layers,text_proj_init,pool,freeze_text_proj,fusion_dim,roc_auc_mean,roc_auc_std,accuracy_mean,accuracy_std,f1_mean,f1_std,n_seeds
0,128,3,3,xavier,sum,True,32,0.879589,0.000815,0.782895,0.038736,0.803593,0.040078,3
1,128,3,2,xavier,sum,True,32,0.875604,0.012895,0.765351,0.031475,0.782107,0.034200,3
2,128,3,4,xavier,sum,True,32,0.864553,0.007521,0.769737,0.037602,0.790371,0.048808,3
3,128,4,3,xavier,sum,True,32,0.858213,0.007032,0.776316,0.014212,0.799949,0.013404,3
4,128,4,4,xavier,sum,True,32,0.840519,0.019093,0.776316,0.023415,0.806961,0.025013,3
5,128,4,2,xavier,sum,True,32,0.825543,0.010761,0.763158,0.010743,0.797198,0.014855,3


In [ ]:
# === Grid Search Ergebnisse speichern ===
out_path = workspace_root / "benchmarking" / "results" / "grid_search_bace_4.csv"
grid_df.to_csv(out_path, index=False)
print(f"\u2713 Saved to {out_path}")

✓ Saved to c:\Users\robsc\Home\Dev\molfusion2\benchmarking\results\grid_search_bace_3.csv
